# EM Induction with the GEM-2: Reading, Checking and Mapping Conductivity

**Near-Surface Geophysics — Missouri S&T**

An EM induction instrument (a "ground-conductivity meter") has two coils.
The **transmitter** drives an alternating current, and its magnetic field
(the **primary** field) induces small eddy currents in the ground. Those
currents make a **secondary** field, and the **receiver** coil measures it
as a fraction of the primary — in **parts per million (ppm)**. No electrodes,
no contact with the ground: you walk, tow or fly it.

The secondary field has two parts:

- **quadrature** (Q), 90° out of phase with the transmitter: mostly
  **electrical conductivity** — clay, water, salt, and the dissolved salts
  in pore water;
- **in-phase** (I), in step with the transmitter: mostly **magnetic
  susceptibility** and **metal**.

The Geophex **GEM-2** is a 1.83 m ski with the coils 1.66 m apart that
reads several frequencies at once (here six, 450 Hz to 63 kHz). It can be
**carried by hand** on a shoulder strap, **pulled on a sled**, or **flown
under a drone**. The readings depend as much on how high the ski was as on
the ground, so this notebook starts by recording how the survey was made.

This notebook:

1. **Reads** a GEM-2 export, with the platform it was carried on.
2. **Checks** the acquisition: sample rate, walking speed, GPS, power-line noise.
3. **Looks at the raw channels**, finds the frequencies that are above the
   noise, and removes spikes.
4. Shows **what the frequency does and does not do**, and **why the platform
   height matters**.
5. Turns the quadrature into **apparent conductivity** and checks it against
   the GEM-2's own numbers.
6. **Maps** conductivity and in-phase, and judges the instrument software's
   layered inversion.

**Units.** Metres, seconds, ppm. Conductivity in millisiemens per metre
(mS/m; 1 mS/m = 1000 Ω·m, 20 mS/m = 50 Ω·m).

## 0. Setup

Run this cell first. It installs `shallowgeo` if it is missing (or out of
date) and imports what the notebook uses.

In [ ]:
# Checks for the *EM* code specifically: an older install imports fine and
# then lacks it, and since the version string does not change between builds,
# a plain `pip install` reports "already satisfied".
import importlib
import sys

PACKAGE = ("shallow-geophysics @ "
           "git+https://github.com/Maurer-GEMLab/shallow-geophysics.git")

def shallowgeo_ready():
    for name in [n for n in sys.modules
                 if n == "shallowgeo" or n.startswith("shallowgeo.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    try:
        from shallowgeo.em import add_apparent_conductivity  # noqa: F401
        from shallowgeo.drivers.gem2 import read_gem2  # noqa: F401
        from shallowgeo.datasets import example_data  # noqa: F401
        example_data("gem2-038-em-lines", download=False)
        return True
    except (ImportError, KeyError):
        return False
    except FileNotFoundError:
        return True       # code is current; the data will download below

if not shallowgeo_ready():
    %pip install -q --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    %pip install -q --force-reinstall --no-deps --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    raise ImportError(
        "shallowgeo.em is still missing after installing. "
        "In Colab: Runtime > Restart session, then run this cell again."
    )

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shallowgeo import em
from shallowgeo.em import plotting as emplot
from shallowgeo.drivers.gem2 import read_gem2, read_gem2_gps
from shallowgeo.datasets import example_data

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 110})
pd.set_option("display.width", 120)
pd.set_option("display.precision", 2)
print("ready")

## 1. The data, and how it was collected

Leave `DATA = None` to use the example survey: five short lines walked with
a GEM-2 (`examples/data/gem2-038-em-lines/`). To use your own export, set
`DATA` to the `..._gem.csv` (or `..._gem+inv.csv`) file the Geophex exporter
wrote, and `GPS_LOG` to the `..._gps.csv` next to it if you have it. In
Colab, upload them with the file browser on the left and give their paths,
e.g. `Path("/content/012_7007_gem+inv.csv")`.

### The platform

The file records the frequencies, the readings, the time and the GPS, but
**not how the ski was carried**. You have to say, and the reader will not
guess:

| `PLATFORM` | Nominal height | What else to record |
|---|---|---|
| `"handheld"` | 1.0 m (shoulder strap, ski at hip height) | Your actual height; no metal on the operator; which way each line was walked |
| `"sled"` | 0.25 m — **a placeholder, measure yours** | `TOW_DISTANCE` to the person or vehicle pulling it (keep it ≥ 3–4 m); `GPS_OFFSET` if the antenna is on the vehicle |
| `"drone"` | **none** — you must give heights | Height above ground for **every sample** (altimeter channel, or GPS altitude minus a ground model); `SLING_LENGTH`; motor noise |

Anything you leave as `None` is filled with the platform's nominal value
and listed as **ASSUMED** in the summary, so a report can say which numbers
were measured and which were guessed. The coil orientation matters as much:
`"HCP"` (horizontal coplanar) is the ski lying flat, `"VCP"` (vertical
coplanar) is the ski turned on its edge.

The example was not labelled when it was collected. It was walked
(0.9–1.4 m/s, section 2), and section 7 shows the GEM-2's own conductivity
values are exactly what HCP coils 1 m above the ground give — so it is read
as hand-carried, HCP, at the nominal 1 m.

`DATE`: the export has times of day (UTC, from the GPS) but no date. Give it
if you know it; nothing below needs it.

In [ ]:
DATA = None            # or Path("/content/your_export_gem.csv")
GPS_LOG = None         # optional: the *_gps.csv written with it
DATE = None            # e.g. "2026-11-12"

# How the GEM-2 was carried. The file does not say, and every number below depends on it.
PLATFORM = "handheld"  # "handheld", "sled" or "drone"
SENSOR_HEIGHT = None   # metres above ground; None = the platform's nominal value (marked ASSUMED)
ORIENTATION = "HCP"    # "HCP" = ski flat, "VCP" = ski on its edge
GPS_OFFSET = None      # (forward, right, up) of the GPS antenna from the ski centre, metres
TOW_DISTANCE = None    # sled: metres from the ski to whoever or whatever pulls it
SLING_LENGTH = None    # drone: metres from the aircraft to the ski
DRONE_HEIGHT = None    # drone: name of an altimeter column in the file, if it has one

if DATA is None:
    raw = example_data("gem2-038-em-lines")
    DATA, GPS_LOG = raw / "038_7007_gem+inv.csv", raw / "038_7007_gps.csv"
    PLATFORM, SENSOR_HEIGHT, ORIENTATION = "handheld", None, "HCP"
    print(f"Using the example survey: {DATA.name}\n")

survey = read_gem2(
    DATA, platform=PLATFORM, sensor_height=SENSOR_HEIGHT, orientation=ORIENTATION,
    gps_offset=GPS_OFFSET, tow_distance=TOW_DISTANCE, sling_length=SLING_LENGTH,
    height=DRONE_HEIGHT, date=DATE,
)
print(survey)
print()
print(survey.deployment)
print()
print(survey.deployment.advice)

For a **drone** survey without an altimeter column, the height comes from
the GPS altitude minus the ground elevation (one number for flat ground, or
a function `f(x, y)` that looks up a DEM, in the same vertical datum):

```python
survey = read_gem2(DATA, platform="drone", sling_length=3.0, height=np.full(n, np.nan))
survey = em.heights_from_altitude(survey, ground_elevation=339.8)
```

For a **sled** with the GPS antenna on the tow vehicle, move the positions
back to the ski before mapping, or every anomaly is drawn ahead of where it
is, in opposite directions on lines driven opposite ways:

```python
survey = em.apply_gps_offset(em.add_distance(survey))
```

The instrument's own header line, and how long each line was:

In [ ]:
for k in ("serial", "firmware", "exporter", "logged_file", "samples_in_header"):
    print(f"{k:18s} {survey.metadata[k]}")
print(f"{'samples in file':18s} {len(survey)}")
print(f"{'frequencies (Hz)':18s} {', '.join(f'{f:g}' for f in survey.frequencies)}")
print()
survey = em.add_distance(survey)   # distance along each line, and its heading
survey.summary()

`length_m` is measured along a straight line fitted to each line's track.
The GPS positions scatter by about a metre, so adding up the sample-to-
sample steps instead would make every line several metres longer than it was.

## 2. Acquisition checks

Before looking at the ground, four things that make readings untrustworthy
on their own:

- **sample interval** — steady at the instrument's output rate (here every
  166.7 ms, five 30 Hz base periods stacked); gaps are lost data;
- **speed** — a steady walk; at 1 m/s and 6 samples per second the samples
  are 17 cm apart, far closer than anything a 1.66 m coil pair can resolve;
- **GPS fix** — 2 is differential (SBAS) GPS, good to about a metre; 4 is
  RTK. With lines only a few metres apart, a metre matters;
- **power-line field** — the 60 Hz noise monitor. A rise means a power line
  or buried cable nearby, and noisier low frequencies.

Alternate lines are shaded.

In [ ]:
emplot.plot_qc(survey)
plt.tight_layout()
plt.show()

if GPS_LOG is not None:
    gps = read_gem2_gps(GPS_LOG)
    print(f"{len(gps)} GPS fixes at 1 Hz; fix quality {sorted(gps['quality'].unique())}, "
          f"satellites {gps['n_sats'].min()}-{gps['n_sats'].max()}, "
          f"HDOP {gps['hdop'].min()}-{gps['hdop'].max()}, "
          f"geoid separation {gps['geoid_separation'].iloc[0]} m")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
emplot.plot_track(survey, ax=ax)
ax.set_title("samples by line; arrows show the direction walked", loc="left", fontsize=9)
plt.show()

The example is five roughly parallel lines, 15–23 m long and 2–4 m apart,
walked in alternating directions — a quick test patch rather than a full
grid. With lines that far apart a map can only be interpolated a little way
off each line (section 10), but every step is the same on a proper grid.
The kinks at the ends of lines 3 and 5 are GPS scatter (about a metre with
DGPS), not the operator's path.

## 3. The raw readings

Every frequency, quadrature then in-phase, against sample number. Look for
three things: whether the frequencies move together (the ground does that),
whether a channel is mostly noise, and whether something sudden happens on
every channel at once (that is not the ground).

In [ ]:
emplot.plot_channels(survey, "Q")
plt.tight_layout()
plt.show()
emplot.plot_channels(survey, "I")
plt.tight_layout()
plt.show()

In the example, two things stand out:

- **The start of line 3** (sample 210): the in-phase starts at about
  −38 000 ppm on *every* frequency, climbs back over eight samples (1.3 s),
  overshoots and settles; the quadrature dips with it. Line 2 ended at the
  same spot six seconds earlier with ordinary readings, so nothing buried
  there makes this. It is the instrument settling when logging restarted,
  or the ski being swung round at the turn. It is removed in section 4.
- **Line 1, sample 30**: a one-sample dip on every frequency.
- **The two lowest frequencies are noise.** 450 Hz and 1530 Hz scatter by
  hundreds of ppm with no relation to the higher frequencies. Section 4
  shows why.

## 4. Signal against noise, and spikes

The quadrature from the ground grows with frequency — in proportion to it at
low frequency — so a small coil pair at a few hundred hertz has almost
nothing to measure. `noise_table` estimates each channel's noise from the
sample-to-sample scatter, and, given a typical conductivity, what uniform
ground would read at this height (`expected`). A channel whose `snr` is
near or below 1 cannot tell you about the ground.

In [ ]:
TYPICAL_CONDUCTIVITY = 0.02   # S/m (20 mS/m) -- a guess is fine; it only sets `expected`
table = em.noise_table(survey, conductivity=TYPICAL_CONDUCTIVITY)
table

In the example, the 450 Hz quadrature should read about 30 ppm and its
noise is about 230 ppm; at 1530 Hz it is 100 ppm against 75. Those two
frequencies carry no usable ground signal and are left out below. From
5 kHz up the signal is well above the noise, and 63 kHz is the cleanest.

The in-phase `expected` is small and positive (a few to a few hundred ppm),
but the measured in-phase sits at hundreds to −1900 ppm. Section 8 returns
to that.

**Transients, then spikes.** A transient several samples long at the start
of a line cannot be caught by comparing each sample with its neighbours —
its neighbours are the transient. Drop it by size first: samples whose
in-phase at the highest frequency is more than `TRANSIENT_PPM` from the
survey median. (Choose the number from the plot above. On a survey for
buried metal, a target can be that large too — look at where the dropped
samples are before accepting it.)

Then **despike**: a reading far from the running median of its neighbours —
more than `THRESHOLD` times the channel's noise — is removed from every
channel, since a transient rarely affects only one frequency.

In [ ]:
USE = [f for f in survey.frequencies if table.loc[f"Q_{f:g}", "snr"] > 3]
print("frequencies used from here on:", ", ".join(f"{f:g} Hz" for f in USE))

TRANSIENT_PPM = 5000
icol = em.channel_name("I", max(survey.frequencies))
far = ((survey.readings[icol] - survey.readings[icol].median()).abs() > TRANSIENT_PPM).to_numpy()
label = lambda r: r[["line", "sample"]].astype(str).agg("-".join, axis=1).tolist()
print(f"transient: dropped {far.sum()} samples:", label(survey.readings[far]))
trimmed = survey.select(~far, reason=f"drop |{icol} - median| > {TRANSIENT_PPM} ppm")

THRESHOLD = 6.0
channels = [em.channel_name(c, f) for f in USE for c in ("I", "Q")]
clean = em.despike(trimmed, channels, threshold=THRESHOLD)
print(f"despike: removed {clean.readings['spike'].sum()} samples:",
      label(clean.readings[clean.readings["spike"]]))

## 5. What the frequency does — and does not do

The usual pitch for a multi-frequency sensor is that lower frequencies see
deeper. That is true only when the **skin depth**
$\delta = \sqrt{2/(\omega\mu_0\sigma)} \approx 503\sqrt{\rho/f}$ m is not
much larger than the coil separation $s$. Their ratio is the **induction
number** $B = s/\delta$.

When $B \ll 1$ — the **low induction number (LIN)** range — the quadrature
is simply proportional to conductivity,

$$Q \approx \frac{\omega \mu_0 \sigma s^2}{4}\, R(h/s),$$

and how deep it looks is set by the **coil separation and height**, not the
frequency. Here is $B$ for this instrument over ground of the typical
conductivity:

In [ ]:
sig = TYPICAL_CONDUCTIVITY
pd.DataFrame({
    "frequency (Hz)": survey.frequencies,
    "skin depth (m)": em.skin_depth(sig, survey.frequencies),
    "induction number B": em.induction_number(sig, survey.frequencies, survey.coil_separation),
}).set_index("frequency (Hz)")

Every frequency is in the LIN range ($B < 0.15$; the LIN formula is within
a few percent below about 0.3). Over ground like this, **all six
frequencies look at the same depth** and should give the same apparent
conductivity; the 63 kHz reading is just 140 times larger than the 450 Hz
one, and so 140 times further above the noise. Frequency starts to buy
depth information only over conductive ground (hundreds of mS/m) or with a
much wider coil separation.

**So what does set the depth?** McNeill's cumulative response $R$: the
fraction of the reading that comes from below a given depth. The curves
below are for the GEM-2's 1.66 m, at sled, hand and drone heights.

In [ ]:
emplot.plot_depth_response(survey.coil_separation)
plt.show()

rows = []
for name, h in [("sled", 0.25), ("handheld", 1.0), ("drone", 3.0)]:
    for o in ("HCP", "VCP"):
        rows.append({"platform": name, "height (m)": h, "orientation": o,
                     "signal vs. on the ground": em.lin_height_factor(h, survey.coil_separation, o),
                     "70 % of signal from above (m)": em.depth_of_investigation(survey.coil_separation, o, h)})
pd.DataFrame(rows).set_index(["platform", "orientation"])

Two lessons from the table:

- **Height costs signal.** Hand-carried HCP keeps 64 % of the ground
  signal it would have on the ground; a drone at 3 m keeps 27 %. Noise does
  not shrink with height, so the drone's signal-to-noise ratio is less than
  half the sled's.
- **"Depth of investigation" is relative.** It *grows* with height (3.3 m
  hand-carried, 7.3 m from a drone) — not because the sensor sees farther,
  but because the topsoil no longer dominates a reading that has become
  much weaker. VCP is shallower than HCP at every height.

## 6. Why the platform has to be recorded

The same ground, read from the three platforms. Left: the quadrature at
63 kHz over 20 mS/m ground. Right: the in-phase from soil with a magnetic
susceptibility of $10^{-3}$ SI.

In [ ]:
h = np.linspace(0.05, 4, 200)
f = survey.frequencies.max()
s = survey.coil_separation
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for o, ls in (("HCP", "-"), ("VCP", "--")):
    q = [em.halfspace_response(0.02, f, s, hh, o).imag for hh in h]
    axes[0].plot(h, q, ls, label=o)
    axes[1].plot(h, 1e-3 * em.lin_susceptibility_factor(h, s, o), ls, label=o)
for ax in axes:
    for name, hh in [("sled", 0.25), ("hand", 1.0), ("drone", 3.0)]:
        ax.axvline(hh, color="0.7", lw=0.8)
        ax.text(hh, ax.get_ylim()[1], f" {name}", fontsize=8, va="top")
    ax.axhline(0, color="0.5", lw=0.6)
    ax.set_xlabel("sensor height above ground (m)")
    ax.legend()
axes[0].set_ylabel(f"quadrature at {f/1e3:g} kHz (ppm)")
axes[0].set_title("20 mS/m ground", fontsize=9)
axes[1].set_ylabel("in-phase (ppm)")
axes[1].set_title("susceptibility 1e-3 SI", fontsize=9)
plt.tight_layout()
plt.show()
print(f"HCP in-phase from magnetic soil changes sign at h = 0.354 s = {0.354 * s:.2f} m")

- The quadrature from a hand-carried ski is 0.64 of the sled's over the
  same ground. Process sled data as hand-carried and the conductivities come
  out 1.5 times too high; the other way round, 0.64 times too low.
- In HCP the **in-phase from magnetic soil changes sign** at 0.59 m: a sled
  sees it positive, a hand-carried ski negative. Two surveys of one field,
  one towed and one walked, disagree in sign until the platform is known.

That is why `read_gem2` will not run without `platform=`.

## 7. Apparent conductivity

The **apparent conductivity** is the conductivity of uniform ground that
would give the measured quadrature at this height and orientation. It is
computed from the full solution for a half-space (not just the LIN line),
so it stays right if the ground is conductive.

The example file also has the GEM-2 software's own values (`Econd`).
Comparing them is a check of the deployment: if they agree, the software
assumed the same height and orientation.

In [ ]:
cond = em.add_apparent_conductivity(clean, USE)

fig, axes = plt.subplots(1, len(USE), figsize=(3.2 * len(USE), 3.2))
for ax, f in zip(np.atleast_1d(axes), USE):
    ours = cond.readings[f"cond_{f:g}"]
    col = f"econd_{f:g}_mS_m"
    if col in cond.vendor:
        theirs = cond.vendor[col]
        ax.plot(theirs, ours, ".", ms=3)
        lim = [np.nanmin([ours.min(), theirs.min()]), np.nanmax([ours.max(), theirs.max()])]
        ax.plot(lim, lim, "k-", lw=0.6)
        ax.set_xlabel("GEM-2 software (mS/m)")
        ax.set_title(f"{f/1e3:g} kHz: ratio {np.nanmedian(ours / theirs):.3f}", fontsize=9)
axes[0].set_ylabel(f"shallowgeo, {cond.orientation} at {cond.heights()[0]:.2f} m (mS/m)")
plt.tight_layout()
plt.show()

In the example, the two agree to within 1 % from 5 to 18 kHz: the GEM-2
software assumed **HCP at about 1 m**, which is how the survey is read here.
At 63 kHz ours is 5 % higher — the GEM-2 software's formula is not
published, and at the highest frequency the full solution and an
approximate one part company first. Try `SENSOR_HEIGHT = 0.25` in section 1 and run again:
the conductivities fall by a third and stop agreeing.

## 8. Do the frequencies agree?

All the frequencies are in the LIN range (section 5), so over any ground
they should give nearly the same apparent conductivity. The black points
are the measured median and interquartile range; the dashed lines are what
uniform ground of three conductivities would read.

In [ ]:
emplot.plot_frequency_response(cond, conductivities=(0.005, 0.015, 0.03))
plt.tight_layout()
plt.show()
cond.readings[[f"cond_{f:g}" for f in USE]].median().rename("median apparent conductivity (mS/m)")

They do not. In the example the median apparent conductivity rises from
about 8 mS/m at 5 kHz to 30 mS/m at 63 kHz — a factor of four — and the
in-phase is large and *negative* at every frequency where uniform ground
gives a small positive value.

A layered ground cannot do that in the LIN range, because every frequency
weights the depths the same way. What can is a **zero-level offset**: each
channel's reading includes a constant error from the imperfect cancellation
of the primary field (the bucking coil), which shifts with temperature and
is set by calibration. The GEM-2's internal temperature rose from 31 to 36 °C
during this survey (see the file header), and no free-air (sensor lifted)
reading was taken to measure the offsets.

**The test that separates the two:** if the *variations* along the lines are
the ground, they must scale between frequencies as the physics says,
whatever the offsets. Plot each frequency against 63 kHz; the slope should
match the predicted ratio.

In [ ]:
ref = max(USE)
r = cond.readings
smooth = lambda v: v.rolling(9, center=True, min_periods=3).median()
rows = []
fig, axes = plt.subplots(1, len(USE) - 1, figsize=(3.3 * (len(USE) - 1), 3.2))
for ax, f in zip(np.atleast_1d(axes), [f for f in USE if f != ref]):
    x, y = smooth(r[f"Q_{ref:g}"]), smooth(r[f"Q_{f:g}"])
    ok = x.notna() & y.notna()
    slope = np.polyfit(x[ok], y[ok], 1)[0]
    m = em.halfspace_response(np.array([0.02, 0.025]), [f, ref], cond.coil_separation,
                              cond.heights()[0], cond.orientation).imag
    predicted = (m[1, 0] - m[0, 0]) / (m[1, 1] - m[0, 1])
    ax.plot(x, y, ".", ms=2)
    xx = np.array([x.min(), x.max()])
    ax.plot(xx, y[ok].mean() + predicted * (xx - x[ok].mean()), "r-", lw=1, label="predicted slope")
    ax.set_xlabel(f"Q {ref/1e3:g} kHz (ppm)")
    ax.set_ylabel(f"Q {f/1e3:g} kHz (ppm)")
    ax.legend(fontsize=7)
    rows.append({"frequency": f, "measured slope": slope, "predicted slope": predicted,
                 "correlation": np.corrcoef(x[ok], y[ok])[0, 1]})
plt.tight_layout()
plt.show()
pd.DataFrame(rows).set_index("frequency")

In the example, 18 kHz follows 63 kHz closely (correlation 0.92) with a
slope near the predicted one (0.29 against 0.33); 8.8 kHz follows more
loosely, and at 5 kHz the variations are barely above the noise. The
shortfall in slope is what the drift of section 10 would add, if the
offsets drift by different amounts at different frequencies. So:

- the **pattern** of conductivity along the lines is real, and the highest
  frequencies show it best;
- the **absolute level** is not known to better than the zero-level offsets
  — the honest statement is "relative conductivity", unless a free-air or
  calibration reading was taken (`em.free_air_offsets`,
  `em.subtract_offsets`);
- the vendor's frequency-to-frequency differences are not depth information.

Take a free-air reading — the ski held above your head, well clear of the
ground, for ten seconds — at the start and end of every survey.

## 9. Heading check

When an operator carries the ski, their body (and anything metal on them)
is on one side of it walking one way and the other side walking back. If
lines walked in opposite directions disagree by much more than the noise,
that difference is the operator, not the ground. `ratio` is the difference
in units of the channel's noise.

In [ ]:
em.heading_check(cond, [em.channel_name(c, f) for f in USE for c in ("Q", "I")])

In the example every ratio is below 1: no heading error to remove. (On a
real grid, compare adjacent lines, which cross similar ground; here the
lines are few and crossing, so this is a coarse check.)

## 10. Maps and profiles

Conductivity at the highest usable frequency, sample by sample and gridded.
The grid is blank more than `MAX_DISTANCE` from a sample: with lines a few
metres apart, anything between them is the interpolator's invention.

In [ ]:
F = max(USE)
COL = f"cond_{F:g}"
CELL, MAX_DISTANCE = 0.5, 1.5
vmin, vmax = np.nanpercentile(cond.readings[COL], [2, 98])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
emplot.plot_track(cond, COL, ax=axes[0], vmin=vmin, vmax=vmax, cmap="viridis",
                  label=f"apparent conductivity {F/1e3:g} kHz (mS/m)")
emplot.plot_map(cond, COL, ax=axes[1], cell=CELL, max_distance=MAX_DISTANCE,
                vmin=vmin, vmax=vmax, label=f"apparent conductivity {F/1e3:g} kHz (mS/m)")
plt.tight_layout()
plt.show()

In [ ]:
emplot.plot_profiles(cond, [f"cond_{f:g}" for f in sorted(USE, reverse=True)[:2]]
                     + [em.channel_name("I", F)])
plt.tight_layout()
plt.show()
cond.readings.groupby("line")[[f"cond_{f:g}" for f in USE]].median()

**Line to line.** In the example the line medians fall steadily from line 1
to line 5 at every frequency. The lines are numbered in the order they were
walked, so this is also a trend **with time**: about 1 mS/m per minute over
the 2.5-minute survey, while the instrument's internal temperature rose from
31 to 36 °C. A trend with time is what instrument drift looks like; a trend
across the site is what geology looks like; here they are the same thing and
cannot be separated, because no line was walked twice. Repeat the first
line at the end of a survey and the difference between the two passes *is*
the drift.

In [ ]:
r = cond.readings
t_min = (r["time"] - r["time"].iloc[0]).dt.total_seconds() / 60
per_line = r.assign(t_min=t_min).groupby("line").agg(
    start_min=("t_min", "min"), **{c: (c, "median") for c in [COL, em.channel_name("I", F)]})
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, c in zip(ax, [COL, em.channel_name("I", F)]):
    a.plot(t_min, r[c], ".", ms=2, color="0.7")
    a.plot(per_line["start_min"], per_line[c], "ko-")
    slope = np.polyfit(t_min[r[c].notna()], r[c].dropna(), 1)[0]
    a.set_title(f"{c}: trend {slope:+.2f} per minute", fontsize=9)
    a.set_xlabel("minutes since first sample")
plt.tight_layout()
plt.show()
per_line

**In-phase.** Its absolute level is the zero-level offset (section 8), but
its *variations* are susceptibility and metal. Plotting it relative to the
survey median removes the offset; a buried metal object shows as a sharp
excursion a few samples wide on every frequency at once.

In [ ]:
icol = em.channel_name("I", F)
rel = cond.readings[icol] - cond.readings[icol].median()
lim = np.nanpercentile(np.abs(rel), 98)
fig, ax = plt.subplots(figsize=(7, 5.5))
emplot.plot_track(cond, rel, ax=ax, cmap="RdBu_r", vmin=-lim, vmax=lim,
                  label=f"in-phase {F/1e3:g} kHz minus median (ppm)")
plt.show()

## 11. The GEM-2 software's layered model

The example export (`..._gem+inv.csv`) also carries a three-layer
resistivity model for every sample, from the GEM-2 software. Before drawing
it, look at how well it fits the data it was made from — the software
reports that itself, as `FitError`.

In [ ]:
v = cond.vendor
if "fiterror_pct" in v:
    print(f"fit error: median {v['fiterror_pct'].median():.0f} %, "
          f"range {v['fiterror_pct'].min():.0f}-{v['fiterror_pct'].max():.0f} %")
    print(f"second-layer thickness exactly 10 m on {np.mean(v['t_layer02_m'] == 10):.0%} of samples")
    print(f"distinct first-layer resistivities: {v['r_layer01_ohm_m'].nunique()} "
          f"for {len(v)} samples")
    emplot.plot_vendor_inversion(cond, cond.lines[0])
    plt.tight_layout()
    plt.show()
else:
    print("this export has no layered-inversion columns")

In the example the model misfits its own data by 100–220 % (median about
135 %), the second layer is stuck at 10 m — likely the edge of the search
range — on most samples, and the resistivities come from a small set of
repeated values. A model that does not fit its data says nothing about the
ground, and section 8 explains why it could not: the frequencies differ
mainly by their offsets, and the inversion has fitted those as layers.
**Do not interpret it.**

## 12. Checklist for your own survey

Write these down in the field. The GEM-2 records none of them:

1. **Platform** and **sensor height** — measured, not assumed. For a
   drone, the altimeter or a ground model, and the sling length.
2. **Orientation** (HCP/VCP) and which way the boom pointed relative to
   the direction of travel.
3. **GPS antenna position** relative to the ski (forward, right, up).
4. **Free-air readings** (ski lifted clear of the ground) at the start
   and end, to measure the zero levels.
5. **A repeat line** at the end, over the first line, to measure drift.
6. **Metal nearby**: fences, vehicles, rebar, buried utilities, and on the
   operator.
7. **Frequencies**: keep the highest ones. On a 1.66 m coil pair over
   ordinary soil, frequencies below a few kHz are below the noise.